<a href="https://colab.research.google.com/github/okechpj/E-commerce_analysis/blob/main/Ecommerce_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
!pip install kaggle

### Kaggle API Authentication

To use the Kaggle API, you need to authenticate by providing your API key. Follow these steps:

1.  **Generate an API Key:**
    *   Go to your Kaggle account page (`kaggle.com/your_username/account`).
    *   Scroll down to the 'API' section.
    *   Click on 'Create New API Token'. This will download a `kaggle.json` file to your computer.

2.  **Upload `kaggle.json` to Colab:**
    *   In Colab, click on the 'Files' icon (folder icon) on the left sidebar.
    *   Click on the 'Upload to session storage' icon (looks like an arrow pointing up into a cloud).
    *   Select the `kaggle.json` file you just downloaded and upload it.

3.  **Move `kaggle.json` to the correct directory:**
    *   Run the following Python code cell to move the `kaggle.json` file to the `.kaggle` directory, which the Kaggle API expects.
    *   Make sure to set the correct permissions for security.

In [ ]:
import os

# Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# Move kaggle.json into the .kaggle directory
!mv kaggle.json ~/.kaggle/

# Set permissions (important for security)
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API key configured successfully!")

mv: cannot stat 'kaggle.json': No such file or directory
Kaggle API key configured successfully!


### Downloading a Dataset using Kaggle API

Once authenticated, you can download datasets directly into your Colab environment. You'll need the dataset's 'API Command' from its Kaggle page (e.g., `kaggle datasets download -d username/dataset-name`).

In [ ]:
# To download a dataset like 'ramametuk/cvd-risk-prediction':
!kaggle datasets download -d ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training

# After downloading, you might need to unzip the file
!unzip /content/cafe-sales-dirty-data-for-cleaning-training.zip


Dataset URL: https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training
License(s): CC-BY-SA-4.0
cafe-sales-dirty-data-for-cleaning-training.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  /content/cafe-sales-dirty-data-for-cleaning-training.zip
replace dirty_cafe_sales.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
df = pd.read_csv('/content/dirty_cafe_sales.csv')

In [ ]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [ ]:
# Take a general look at the dataset structure

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


## **Dataset Cleaning**

# 1. Convert columns to the right datatypes

In [ ]:
# Convert quantity column from object to numeric
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').astype('Int64')

In [ ]:
# Convert Price per unit from object to float
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce').astype('float')

In [ ]:
#Convert Total spent column from object to float
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce').astype('float')

In [ ]:
#convert Transaction date to datetime datatype
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

# 2. Missing Values

In [ ]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


**Foward fill the Item column**

In [ ]:
# Replace 'ERROR' and 'UNKNOWN' with NaN first
df['Item'].replace(['ERROR', 'UNKNOWN'], np.nan, inplace=True)

# Foward fill missing Items values
df['Item'].isnull().sum()

#333

np.int64(969)

In [ ]:
df['Item'] = df['Item'].ffill()

**Foward fill Quantity column values**

In [ ]:
## Foward fill the Quantity column
df['Quantity'].isnull().sum()

#479

np.int64(479)

In [ ]:
df['Quantity'] = df['Quantity'].ffill()

### **Fill Price per unit missing values**
Group the items and find their mode prices then fill in the missing values with the modes of those prices **bold text**

In [ ]:
# Calculate the mode of 'Price Per Unit' for each 'Item'
item_price_mode = df.groupby('Item')['Price Per Unit'].apply(lambda x: x.mode()[0] if not x.mode().empty else np.nan)

# Fill missing 'Price Per Unit' values based on the 'Item'
df['Price Per Unit'] = df.apply(lambda row: item_price_mode[row['Item']] if pd.isna(row['Price Per Unit']) and row['Item'] in item_price_mode else row['Price Per Unit'], axis=1)

# Display the count of missing values after filling
df['Price Per Unit'].isnull().sum()

np.int64(0)

In [ ]:
df['Price Per Unit'].isnull().sum()

np.int64(0)

### Fill Missing 'Total Spent' Values

Missing values in the 'Total Spent' column are filled by calculating the product of 'Quantity' and 'Price Per Unit' for each row. This ensures consistency and completes the financial records in the dataset.

In [ ]:
# Fill missing 'Total Spent' values by multiplying 'Quantity' and 'Price Per Unit'
df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])

# Check for remaining missing values in 'Total Spent' column
df['Total Spent'].isnull().sum()

np.int64(0)

In [ ]:
df['Payment Method'].isnull().sum()

np.int64(2579)

### Fill Missing 'Payment Method' Values

To effectively handle missing values in the 'Payment Method' column, a multi-step approach was used:
1.  **Standardize Missing Indicators:** Initially, 'ERROR' and 'UNKNOWN' string values were replaced with `NaN` (Not a Number) to ensure all missing entries are consistently represented.
2.  **Forward Fill (`ffill`):** Missing values were then filled by propagating the last valid observation forward to the next valid observation. This helps to fill gaps where a payment method might have remained consistent over a few transactions.
3.  **Backward Fill (`bfill`):** Finally, any remaining `NaN` values (typically those at the very beginning of the dataset that `ffill` couldn't address) were filled by propagating the next valid observation backward. This ensures that all `NaN` entries are imputed.

In [ ]:
# Replace 'ERROR' and 'UNKNOWN' with NaN first
df['Payment Method'].replace(['ERROR', 'UNKNOWN'], np.nan, inplace=True)

# Forward fill missing values in 'Payment Method' column
df['Payment Method'] = df['Payment Method'].ffill()

# Backward fill any remaining NaN values (e.g., those at the start)
df['Payment Method'] = df['Payment Method'].bfill()

# Verify the changes
df['Payment Method'].value_counts(dropna=False)

/tmp/ipython-input-320768830.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Payment Method'].replace(['ERROR', 'UNKNOWN'], np.nan, inplace=True)


,count
Payment Method,
Credit Card,3365
Digital Wallet,3343
Cash,3292


### Fill Missing 'Location' Values

To handle the numerous missing values, 'ERROR', and 'UNKNOWN' entries in the 'Location' column, a consolidated approach was chosen. All instances of `NaN`, 'ERROR', and 'UNKNOWN' were replaced with the string 'Unknown'. This strategy avoids making speculative imputations and instead explicitly categorizes these entries as unidentifiable locations. It allows for analysis of transactions from unknown origins as a distinct group and maintains data integrity by not introducing potentially incorrect location names.

In [ ]:
# Replace NaN, 'ERROR', and 'UNKNOWN' values in 'Location' with 'Unknown'
df['Location'].replace(['ERROR', 'UNKNOWN', np.nan], 'Unknown', inplace=True)

# Verify the changes
df['Location'].value_counts(dropna=False)

/tmp/ipython-input-1388575349.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Location'].replace(['ERROR', 'UNKNOWN', np.nan], 'Unknown', inplace=True)


,count
Location,
Unknown,3961
Takeaway,3022
In-store,3017


In [ ]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,Credit Card,Unknown,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,Cash,Unknown,2023-08-30
9996,TXN_9659401,Coffee,3,2.0,3.0,Digital Wallet,Unknown,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,Unknown,2023-03-02
9998,TXN_7695629,Cookie,3,1.0,3.0,Digital Wallet,Unknown,2023-12-02


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              10000 non-null  object        
 2   Quantity          10000 non-null  Int64         
 3   Price Per Unit    10000 non-null  float64       
 4   Total Spent       10000 non-null  float64       
 5   Payment Method    10000 non-null  object        
 6   Location          10000 non-null  object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: Int64(1), datetime64[ns](1), float64(2), object(4)
memory usage: 634.9+ KB


### Fill Missing 'Transaction Date' Values

To handle missing values in the 'Transaction Date' column, a two-step imputation process was applied:
1.  **Forward Fill (`ffill`):** Missing date entries were initially filled by propagating the last valid observation forward. This is effective for maintaining chronological order and is suitable when dates are likely to be consecutive or repeat.
2.  **Backward Fill (`bfill`):** Any remaining `NaN` values, typically those at the very beginning of the dataset that `ffill` could not address, were then filled by propagating the next valid observation backward. This ensures that all missing date entries are imputed while preserving the temporal context.

In [ ]:
print(f"Missing values in 'Transaction Date': {df['Transaction Date'].isnull().sum()}")
print(f"Data type of 'Transaction Date': {df['Transaction Date'].dtype}")

Missing values in 'Transaction Date': 460
Data type of 'Transaction Date': datetime64[ns]


In [ ]:
# Apply forward fill to 'Transaction Date'
df['Transaction Date'] = df['Transaction Date'].ffill()

# Apply backward fill to catch any remaining NaNs (e.g., at the start of the series)
df['Transaction Date'] = df['Transaction Date'].bfill()

# Verify the changes
print(f"Missing values in 'Transaction Date' after fill: {df['Transaction Date'].isnull().sum()}")

Missing values in 'Transaction Date' after fill: 0


## **3. Deal with Duplicates**

In [ ]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,Credit Card,Unknown,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,Cash,Unknown,2023-08-30
9996,TXN_9659401,Coffee,3,2.0,3.0,Digital Wallet,Unknown,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,Unknown,2023-03-02
9998,TXN_7695629,Cookie,3,1.0,3.0,Digital Wallet,Unknown,2023-12-02


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
#### No duplicates present in the dataset

# **4. Standardize column names**

Standardize the column names by removing the extra spaces, converting to lower case, replace special characters and replace spaces with underscore

In [ ]:
import re

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'[^\w\s]', '', regex=True)
      .str.replace(r'\s+', '_', regex=True)
)


## **5. Export dataset to a CSV document**

In [ ]:
df.to_csv("cleaned_cafe_dataset.csv", index=False)
